# 06 — Limpieza profesional con criterio — Airbnb Listings

La diferencia entre limpieza mecánica y limpieza profesional no está en las funciones que se usan, sino en el orden en que se aplican y en las decisiones que se toman en cada paso.

La limpieza mecánica aplica `dropna()`, `fillna()` y `drop_duplicates()` en el orden que sea, sin preguntarse si tiene sentido hacerlo así. La limpieza profesional sigue un orden fijo con una justificación detrás de cada decisión.

**El orden correcto es siempre este, y no es intercambiable:**

1. **Tipos correctos** — sin esto, las detecciones de nulos y duplicados pueden fallar en silencio
2. **Duplicados** — antes de rellenar nulos para no fabricar datos en filas que se van a eliminar
3. **Nulos con criterio** — cada columna puede necesitar una decisión distinta
4. **Inconsistencias de formato** — lo último porque depende de tener los tipos correctos primero

Al terminar este notebook se tienen los datos en un estado fiable para cualquier análisis posterior.

## Setup

Se carga el dataset con `encoding='latin-1'` porque el archivo contiene caracteres especiales de múltiples ciudades del mundo (tildes, ñ, caracteres del árabe, tailandés, etc.) que no son válidos en UTF-8. Usar la codificación incorrecta lanzaría un `UnicodeDecodeError` al leer el archivo.

In [1]:
import pandas as pd
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT = find_project_root()

# low_memory=False evita que pandas lea el archivo en chunks e infiera tipos distintos
# por columna en cada chunk — puede producir columnas con tipos mezclados sin aviso
df = pd.read_csv(ROOT / 'data' / 'external' / 'Listings.csv',
                 encoding='latin-1', low_memory=False)

print(f'Shape: {df.shape}')
print(df.dtypes)


Shape: (279712, 33)
listing_id                       int64
name                               str
host_id                          int64
host_since                         str
host_location                      str
host_response_time                 str
host_response_rate             float64
host_acceptance_rate           float64
host_is_superhost                  str
host_total_listings_count      float64
host_has_profile_pic               str
host_identity_verified             str
neighbourhood                      str
district                           str
city                               str
latitude                       float64
longitude                      float64
property_type                      str
room_type                          str
accommodates                     int64
bedrooms                       float64
amenities                          str
price                            int64
minimum_nights                   int64
maximum_nights                   int64
revie

---
## 1 — Tipos correctos

Este es el primer paso obligatorio antes de cualquier otra operación sobre el dataset.

Cuando una columna almacena un tipo incorrecto — por ejemplo una fecha guardada como texto, o un booleano guardado como los caracteres `'t'` y `'f'` — las operaciones no siempre fallan. Muchas veces devuelven un resultado silenciosamente incorrecto, sin lanzar ningún error.

Por ejemplo:
- `df[df['host_is_superhost'] == True]` sobre una columna con strings `'t'`/`'f'` devuelve un DataFrame vacío, sin error. El filtro simplemente no encuentra ningún `True`.
- `df['host_since'].dt.year` sobre una columna con fechas como strings lanza `AttributeError`, pero `df[df['host_since'] > '2020-01-01']` funciona "por casualidad" porque las fechas en formato ISO 8601 (`2024-11-28`) se ordenan correctamente como texto. Si el formato fuera `28/11/2024`, la comparación sería incorrecta sin aviso.

En este dataset hay dos correcciones necesarias:

1. **`host_since`** — llega como string ISO 8601 y se convierte a `datetime64` para poder calcular antigüedad, filtrar por rango de fechas, o agrupar por año y mes.
2. **Columnas booleanas** — Airbnb exporta los valores `True`/`False` como los strings `'t'` y `'f'`. Se convierten a `bool` para que operen correctamente en filtros y agregaciones.

In [2]:
# Siempre operar sobre una copia del DataFrame original
# Si algo sale mal en la limpieza se puede volver al df original sin recargar el archivo
df = df.copy()

# host_since llega como string '2011-12-03' (formato ISO 8601)
# pd.to_datetime lo convierte a datetime64, que permite operaciones de fecha:
# dt.year, dt.month, diferencia entre fechas, groupby por periodo, etc.
df['host_since'] = pd.to_datetime(df['host_since'])

# Columnas que Airbnb exporta como 't' y 'f' en vez de True y False
bool_cols = ['host_is_superhost', 'host_has_profile_pic',
             'host_identity_verified', 'instant_bookable']

for col in bool_cols:
    # map() reemplaza cada valor según el diccionario
    # Cualquier valor que no sea 't' ni 'f' queda como NaN — correcto,
    # porque hay filas con nulos en estas columnas que deben seguir siendo NaN
    df[col] = df[col].map({'t': True, 'f': False})

print('Tipos corregidos:')
print(df[['host_since'] + bool_cols].dtypes)
print()
print('Distribución superhost:')
print(df['host_is_superhost'].value_counts(dropna=False))


Tipos corregidos:
host_since                datetime64[us]
host_is_superhost                 object
host_has_profile_pic              object
host_identity_verified            object
instant_bookable                    bool
dtype: object

Distribución superhost:
host_is_superhost
False    229294
True      50253
NaN         165
Name: count, dtype: int64


---
## 2 — Duplicados

Los duplicados se eliminan **antes** de tratar los nulos, no después. El orden importa.

Si se rellena un nulo primero y luego se buscan duplicados, puede ocurrir lo siguiente: dos filas que representan el mismo registro tienen valores idénticos en todas las columnas menos en una, que en una fila es nulo y en la otra tiene un valor real. Al rellenar el nulo con la mediana, ahora las dos filas son distintas entre sí aunque representen el mismo alojamiento, y `drop_duplicates()` no las elimina. El resultado es un dataset con un registro duplicado donde uno de los dos tiene un valor fabricado.

Hacerlo en el orden correcto — duplicados primero — garantiza que solo se trabaja con filas únicas antes de tomar cualquier decisión de imputación.

En este dataset se verifican dos cosas:
- Filas completamente duplicadas (todas las columnas iguales)
- `listing_id` duplicado con datos distintos (dos versiones del mismo alojamiento en el export)

In [3]:
# duplicated() devuelve una Series de True/False — True marca las filas repetidas
# .sum() cuenta los True porque True equivale a 1
n_dup = df.duplicated().sum()
print(f'Filas duplicadas: {n_dup}')

# Verificar también duplicados por listing_id — puede haber filas distintas
# con el mismo id si el export tiene versiones del mismo alojamiento
n_dup_id = df.duplicated(subset='listing_id').sum()
print(f'listing_id duplicados: {n_dup_id}')

if n_dup > 0:
    df = df.drop_duplicates()
    print(f'Shape tras drop_duplicates: {df.shape}')
else:
    print('Sin duplicados — no se modifica el DataFrame')


Filas duplicadas: 0
listing_id duplicados: 0
Sin duplicados — no se modifica el DataFrame


---
## 3 — Nulos: diagnóstico

Antes de decidir qué hacer con los nulos, hay que ver el mapa completo de cuántos hay y en qué columnas. No todos los nulos se tratan de la misma manera, y la decisión correcta depende del porcentaje de nulos, del tipo de columna, y de lo que representa la ausencia del dato en ese contexto concreto.

El patrón `df.isnull().mean()` devuelve la proporción de nulos por columna. `isnull()` convierte cada celda en `True` (es nulo) o `False` (tiene valor), y `mean()` sobre valores `True`/`False` da la proporción de `True`, que es exactamente el porcentaje de nulos expresado en decimal.

In [4]:
nulos = (df.isnull().mean() * 100).round(1).sort_values(ascending=False)
print('Porcentaje de nulos por columna:')
print(nulos[nulos > 0].to_string())


Porcentaje de nulos por columna:
district                       86.8
host_response_time             46.0
host_response_rate             46.0
host_acceptance_rate           40.4
review_scores_accuracy         32.8
review_scores_value            32.8
review_scores_location         32.8
review_scores_checkin          32.8
review_scores_communication    32.8
review_scores_cleanliness      32.8
review_scores_rating           32.7
bedrooms                       10.5
host_location                   0.3
host_total_listings_count       0.1
name                            0.1
host_is_superhost               0.1
host_since                      0.1
host_identity_verified          0.1
host_has_profile_pic            0.1


### 3.1 — Columnas con más del 50% de nulos → eliminar la columna

Cuando una columna tiene más de la mitad de sus valores ausentes, su capacidad informativa es muy baja. Rellenarla con la mediana o la moda significaría inventar más datos de los que existen realmente, y eso introduce sesgo en cualquier análisis posterior.

En este dataset, `district` tiene un 86.8% de nulos. Solo 1 de cada 8 filas tiene valor. No tiene sentido conservarla.

Se define `UMBRAL_NULOS` como variable en vez de escribir `0.50` directamente en el código porque si en el futuro se quiere ajustar el criterio (por ejemplo a 0.60 o 0.40), se cambia en un único lugar y el resto del código se adapta solo.

In [5]:
UMBRAL_NULOS = 0.50
cols_a_eliminar = [c for c in df.columns if df[c].isnull().mean() > UMBRAL_NULOS]
print(f'Columnas con >{UMBRAL_NULOS*100:.0f}% nulos: {cols_a_eliminar}')

df = df.drop(columns=cols_a_eliminar)
print(f'Shape tras eliminar columnas vacías: {df.shape}')


Columnas con >50% nulos: ['district']
Shape tras eliminar columnas vacías: (279712, 32)


### 3.2 — `bedrooms` (10.5% nulos) → rellenar con mediana por tipo de propiedad

Con un 10.5% de nulos, eliminar las filas haría perder casi 30.000 registros. La opción correcta es imputar con la mediana — pero no con la mediana global del dataset.

Un `Entire apartment` tiene típicamente 1 o 2 habitaciones. Un `Private room in house` probablemente tiene 3 o 4 (la casa completa tiene más, aunque el huésped solo alquila una habitación). Un `Entire house` puede tener 4 o más. Rellenar todos con la misma mediana global mezclaría estas distribuciones distintas y asignaría valores incorrectos.

La solución es calcular la mediana **dentro de cada tipo de propiedad** y usarla solo para los nulos de ese mismo tipo. `groupby().transform('median')` hace exactamente esto: devuelve una Serie con el mismo índice que el DataFrame original, donde cada posición tiene la mediana del grupo al que pertenece esa fila. La alineación es automática.

In [6]:
mediana_por_tipo = df.groupby('property_type')['bedrooms'].transform('median')
# transform devuelve una Series con el mismo índice que df — alineación automática

nulos_antes = df['bedrooms'].isnull().sum()
df['bedrooms'] = df['bedrooms'].fillna(mediana_por_tipo)

# Los property_type muy raros pueden no tener mediana — rellenar el residuo con la global
df['bedrooms'] = df['bedrooms'].fillna(df['bedrooms'].median())

nulos_despues = df['bedrooms'].isnull().sum()
print(f'Nulos bedrooms: {nulos_antes} → {nulos_despues}')


Nulos bedrooms: 29435 → 0


### 3.3 — `host_response_rate` y `host_acceptance_rate` (40–46% nulos)

Casi la mitad de los hosts no tienen valor en estas columnas. Antes de decidir qué hacer, hay que entender por qué faltan: no es que Airbnb no tenga el dato, es que el host todavía no tiene suficiente historial para calcular una tasa.

Rellenar con `0` sería incorrecto — implicaría que esos hosts no responden nunca, cuando en realidad simplemente son nuevos. Rellenar con la media global también sería impreciso, porque la tasa de respuesta varía mucho entre mercados: un host en Tokio no tiene el mismo comportamiento promedio que uno en Rio de Janeiro.

La decisión es rellenar con la **mediana del grupo de ciudad**. Si la ciudad tampoco tiene suficientes valores para calcular una mediana (ciudades muy pequeñas en el dataset), se usa la mediana global como último recurso.

In [7]:
for col in ['host_response_rate', 'host_acceptance_rate']:
    mediana_ciudad = df.groupby('city')[col].transform('median')
    nulos_antes = df[col].isnull().sum()
    df[col] = df[col].fillna(mediana_ciudad)
    # Residuo — ciudades con muy pocos registros sin valor
    df[col] = df[col].fillna(df[col].median())
    print(f'{col}: {nulos_antes:,} nulos → {df[col].isnull().sum():,}')


host_response_rate: 128,782 nulos → 0
host_acceptance_rate: 113,087 nulos → 0


### 3.4 — `review_scores_*` (32.8% nulos) → mantener como NaN

Esta es la decisión más importante del notebook porque va en contra del instinto de "arreglarlo todo": a veces la decisión correcta es no hacer nada.

Los nulos en las columnas de puntuación no son errores. Son listings que todavía no han recibido ninguna reseña porque son nuevos en la plataforma. Si se rellena un `review_scores_rating` nulo con la mediana de las demás puntuaciones, se está asignando una valoración a un alojamiento que nadie ha puntuado todavía. Eso contamina cualquier análisis de calidad o satisfacción.

La celda de código valida esta hipótesis: compara el comportamiento de los listings con y sin reseñas en `minimum_nights` para confirmar que son grupos distintos y que la ausencia de dato no es aleatoria.

In [8]:
review_cols = [c for c in df.columns if c.startswith('review_scores')]
print('Columnas de reviews — nulos justificados (listings sin historial):')
print(df[review_cols].isnull().sum().to_string())
print()
# Verificar: ¿los nulos en reviews coinciden con listings sin host_since reciente?
sin_review = df[df['review_scores_rating'].isnull()]
con_review  = df[df['review_scores_rating'].notna()]
print(f'Media noches mínimas — sin review: {sin_review["minimum_nights"].median():.0f}')
print(f'Media noches mínimas — con review:  {con_review["minimum_nights"].median():.0f}')


Columnas de reviews — nulos justificados (listings sin historial):
review_scores_rating           91405
review_scores_accuracy         91713
review_scores_cleanliness      91665
review_scores_checkin          91771
review_scores_communication    91687
review_scores_location         91775
review_scores_value            91785

Media noches mínimas — sin review: 2
Media noches mínimas — con review:  2


---
## 4 — Inconsistencias de formato

Una vez que los tipos son correctos y los nulos están tratados, el último paso es normalizar el contenido textual. Pandas distingue entre `'Madrid'` y `'madrid'` como dos valores diferentes, y entre `' Within an hour'` y `'within an hour'` también. Si no se normalizan, un `groupby` o un `value_counts` producirá categorías duplicadas que en realidad son la misma.

### 4.1 — `host_response_time` → normalizar categorías de texto

Esta columna tiene cuatro valores posibles más nulos. El problema no es que haya categorías incorrectas, sino que podrían tener mayúsculas o espacios distintos según la región o la versión del export. Normalizar a minúsculas y sin espacios extra garantiza que las comparaciones y agrupaciones funcionen siempre igual independientemente del origen del archivo.

In [9]:
print('Valores únicos host_response_time:')
print(df['host_response_time'].value_counts(dropna=False))

# Normalizar a minúsculas y sin espacios extra para comparaciones seguras
df['host_response_time'] = (
    df['host_response_time']
    .str.strip()
    .str.lower()
)
print()
print('Tras normalización:')
print(df['host_response_time'].value_counts(dropna=False))


Valores únicos host_response_time:
host_response_time
NaN                   128782
within an hour         83464
within a few hours     28891
within a day           23425
a few days or more     15150
Name: count, dtype: int64

Tras normalización:
host_response_time
NaN                   128782
within an hour         83464
within a few hours     28891
within a day           23425
a few days or more     15150
Name: count, dtype: int64


### 4.2 — `amenities` → convertir string a métrica numérica

La columna `amenities` llega como un string que representa una lista de servicios del alojamiento: wifi, cocina, lavadora, calefacción, etc. En ese formato no se puede usar directamente en un análisis numérico.

En vez de intentar parsear el JSON (cuyo formato puede variar entre filas y entre versiones del dataset), se cuenta el número de elementos separados por comas. El resultado es una columna numérica `amenities_count` que representa cuántos servicios tiene cada alojamiento, y que ya puede usarse en correlaciones, agrupaciones o modelos.

In [10]:
# Muestra del formato original
print(df['amenities'].iloc[0])
print()

# Contar elementos — split por coma aproxima el número de amenities
# (no parsear como JSON porque el formato puede variar entre filas)
df['amenities_count'] = (
    df['amenities']
    .fillna('')
    .str.split(',')
    .apply(len)
)

print('Distribución de amenities_count:')
print(df['amenities_count'].describe().round(1))


["Heating", "Kitchen", "Washer", "Wifi", "Long term stays allowed"]

Distribución de amenities_count:
count    279712.0
mean         19.6
std           9.3
min           1.0
25%          13.0
50%          18.0
75%          26.0
max          89.0
Name: amenities_count, dtype: float64


---
## 5 — Función de limpieza reutilizable

Toda la lógica anterior se encapsula en una función por dos razones:

**Reproducibilidad** — si se necesita volver al dataset limpio en cualquier punto del análisis, basta con llamar a `limpiar_airbnb(df_raw)` en vez de reejecutar 10 celdas en orden. Esto también protege contra el error de ejecutar las celdas fuera de orden.

**Parametrización** — el umbral de nulos es configurable. Si en un análisis concreto se quiere ser más o menos estricto con las columnas vacías, se pasa el parámetro en vez de cambiar el código interno.

La función imprime un resumen al final para que quede registrado en la salida del notebook qué columnas se eliminaron y cuál es el estado final del dataset.

In [11]:
def limpiar_airbnb(df: 'pd.DataFrame', umbral_nulos: float = 0.50) -> 'pd.DataFrame':
    df = df.copy()

    # Tipos
    df['host_since'] = pd.to_datetime(df['host_since'])
    for col in ['host_is_superhost', 'host_has_profile_pic',
                'host_identity_verified', 'instant_bookable']:
        if col in df.columns:
            df[col] = df[col].map({'t': True, 'f': False})

    # Columnas con demasiados nulos
    cols_vacias = [c for c in df.columns if df[c].isnull().mean() > umbral_nulos]
    df = df.drop(columns=cols_vacias)

    # Duplicados
    df = df.drop_duplicates()

    # Nulos
    if 'bedrooms' in df.columns:
        df['bedrooms'] = df['bedrooms'].fillna(
            df.groupby('property_type')['bedrooms'].transform('median')
        ).fillna(df['bedrooms'].median())

    for col in ['host_response_rate', 'host_acceptance_rate']:
        if col in df.columns:
            df[col] = df[col].fillna(
                df.groupby('city')[col].transform('median')
            ).fillna(df[col].median())

    # Formato
    if 'host_response_time' in df.columns:
        df['host_response_time'] = df['host_response_time'].str.strip().str.lower()

    if 'amenities' in df.columns:
        df['amenities_count'] = df['amenities'].fillna('').str.split(',').apply(len)

    print(f'Dataset limpio: {df.shape[0]:,} filas, {df.shape[1]} columnas')
    print(f'Eliminadas: {cols_vacias}')
    return df


# Cargar de nuevo y aplicar la función completa
df_raw = pd.read_csv(ROOT / 'data' / 'external' / 'Listings.csv',
                     encoding='latin-1', low_memory=False)
df_clean = limpiar_airbnb(df_raw)
print()
print('Nulos restantes:')
print((df_clean.isnull().mean() * 100).round(1).sort_values(ascending=False).head(10))


Dataset limpio: 279,712 filas, 33 columnas
Eliminadas: ['district']

Nulos restantes:
host_response_time             46.0
review_scores_location         32.8
review_scores_value            32.8
review_scores_accuracy         32.8
review_scores_cleanliness      32.8
review_scores_checkin          32.8
review_scores_communication    32.8
review_scores_rating           32.7
host_location                   0.3
host_has_profile_pic            0.1
dtype: float64


---
## Resumen de decisiones

| Columna | Nulos | Decisión | Justificación |
|---------|-------|----------|--------------|
| `district` | 86.8% | Eliminar columna | >50% — rellenar sería invención |
| `bedrooms` | 10.5% | Mediana por `property_type` | La mediana global mezcla tipos de propiedad distintos |
| `host_response_rate` | 46% | Mediana por ciudad | Los nulos son hosts sin historial, no errores |
| `review_scores_*` | 32.8% | Mantener NaN | Ausencia justificada: listings sin reviews aún |
| `host_is_superhost` | t/f → bool | Corrección de tipo | Los strings no operan como booleanos |
| `host_since` | str → datetime | Corrección de tipo | Sin datetime no se puede calcular antigüedad |
